# Feature exploration

Start with raw play-by-play data to explore candidate features and preprocessing needs across 2015–2025.

In [12]:
import numpy as np
import pandas as pd
from IPython.display import display
from nfl_project.data import (
    FEATURE_COLUMNS, TARGET_COLUMN, load_data, process_4th_down_data,
    split_data, make_preprocessor,
)

seasons = list(range(2015, 2026))
raw_data = load_data(seasons)
print(f"Raw data: {raw_data.shape[0]:,} rows and {raw_data.shape[1]} columns")

Raw data: 532,376 rows and 372 columns


### Fourth-down plays

First, we should filter to fourth-down plays only. And we want to exclude punts, field goals, penalties, etc., looking only at plays where an offensive play was run. Manually keep `down == 4` and `play_type` in `run`/`pass`.

In [13]:
# Manually select fourth-down runs and passes from the raw data
fourth_down = raw_data.loc[
    raw_data["down"].eq(4) & raw_data["play_type"].isin(["run", "pass"]) & raw_data["penalty"].eq(0)
].copy()
print(f"Fourth-down runs and passes: {len(fourth_down):,}")
display(fourth_down["play_type"].value_counts().rename("plays"))

Fourth-down runs and passes: 7,433


play_type
pass    4708
run     2725
Name: plays, dtype: int64

## 1. Candidate features

There are 372 columns, but we will use our football domain knowledge to select a set of potential features. We want features that do not leak the outcome of the play, so we limit to pre-snap information. Focus on game situation, teams, stadium conditions, and season context for now.

In [14]:
base_candidates = [
    "ydstogo", "yardline_100", "goal_to_go",
    "qtr", "game_seconds_remaining", "half_seconds_remaining",
    "posteam_timeouts_remaining", "defteam_timeouts_remaining",
    "score_differential", "posteam", "defteam", "posteam_type",
    "roof", "surface", "temp", "wind",
    "week", "season_type", "season",
]

### Engineered situation features

Add red-zone (within 20 yards of the endzone), short-yardage (at most 2 yards to go), final-two-minute regulation-half, late-game deficit (trailing in Q4 with at most 5 minutes left), and overtime indicators. Also try log-distance, distance interactions, and timeout advantage.

In [15]:
# Nullable numeric values preserve missing inputs in engineered features
situation = fourth_down[[
    "yardline_100", "ydstogo", "qtr", "half_seconds_remaining",
    "game_seconds_remaining", "score_differential", "goal_to_go",
    "posteam_timeouts_remaining", "defteam_timeouts_remaining",
]].astype("Float64")
engineered = pd.DataFrame(index=fourth_down.index)
engineered["red_zone"] = situation["yardline_100"].between(0, 20).astype("Int64")
engineered["short_yardage"] = situation["ydstogo"].between(0, 2).astype("Int64")
engineered["end_of_half"] = (
    (situation["qtr"].eq(2) | situation["qtr"].eq(4))
    & situation["half_seconds_remaining"].between(0, 120)
).astype("Int64")
engineered["late_game_deficit"] = (
    situation["qtr"].eq(4)
    & situation["game_seconds_remaining"].between(0, 300)
    & situation["score_differential"].lt(0)
).astype("Int64")
engineered["overtime"] = situation["qtr"].ge(5).astype("Int64")
engineered["log_ydstogo"] = np.log1p(situation["ydstogo"])
engineered["distance_x_goal_to_go"] = situation["ydstogo"] * situation["goal_to_go"]
engineered["distance_x_red_zone"] = situation["ydstogo"] * engineered["red_zone"]
engineered["timeout_advantage"] = (
    situation["posteam_timeouts_remaining"] - situation["defteam_timeouts_remaining"]
)

# Add features to the manually selected raw fourth-down rows
eda = fourth_down.join(engineered)
candidates = base_candidates + engineered.columns.tolist()
features = eda[candidates]

## 2. Quick checks

Next, we'll take a look at coverage and variation, as well as correlation with the target (fourth down conversion). `n_with_outcome` reports nonmissing feature values in that subset. Correlations do not capture nonlinear effects or interactions and blank correlations indicate categorical or constant fields.

In [16]:
# Explicit exploratory subset for outcome comparisons; no penalty filter yet.
success = eda["fourth_down_converted"]
failure = eda["fourth_down_failed"]
clear_outcome = success.isin([0, 1]) & failure.isin([0, 1]) & (success + failure).eq(1)
conversion_data = eda.loc[eda["play_type"].isin(["run", "pass"]) & clear_outcome].copy()
print(f"Rows with conversion outcomes: {len(conversion_data):,}")
print(f"Of these, flagged penalties: {conversion_data['penalty'].eq(1).sum():,}")

numeric = conversion_data[candidates].select_dtypes(include="number")
numeric = numeric.loc[:, numeric.nunique() > 1]
correlations = numeric.corrwith(conversion_data["fourth_down_converted"])
summary = pd.DataFrame({
    "dtype": features.dtypes.astype(str),
    "missing_pct": features.isna().mean().mul(100),
    "n_available": features.notna().sum(),
    "unique_values": features.nunique(),
    "n_with_outcome": conversion_data[candidates].notna().sum(),
    "target_correlation": correlations.reindex(candidates),
})
summary.sort_values(
    "target_correlation", key=lambda values: values.abs(), ascending=False,
).round(3)

Rows with conversion outcomes: 7,433
Of these, flagged penalties: 0


,dtype,missing_pct,n_available,unique_values,n_with_outcome,target_correlation
log_ydstogo,Float64,0.000,7433,34,7433,-0.299
ydstogo,float64,0.000,7433,34,7433,-0.285
short_yardage,Int64,0.000,7433,2,7433,0.250
half_seconds_remaining,float64,0.000,7433,1631,7433,0.150
game_seconds_remaining,float64,0.000,7433,2695,7433,0.138
late_game_deficit,Int64,0.000,7433,2,7433,-0.132
qtr,float64,0.000,7433,5,7433,-0.121
end_of_half,Int64,0.000,7433,2,7433,-0.119
posteam_timeouts_remaining,float64,0.000,7433,4,7433,0.116
defteam_timeouts_remaining,float64,0.000,7433,4,7433,0.102


In [17]:
# Compare conversion rates and sample sizes for situation indicators.
indicators = ["goal_to_go", "red_zone", "short_yardage", "end_of_half", "late_game_deficit", "overtime"]
rows = []
for column in indicators:
    rates = conversion_data.groupby(column)["fourth_down_converted"].agg(attempts="size", conversion_rate="mean")
    rates["feature"] = column
    rows.append(rates.rename_axis("value").reset_index())
pd.concat(rows, ignore_index=True).set_index(["feature", "value"]).round(3)

attempts  conversion_rate
feature           value                           
goal_to_go        0.0        6730            0.532
                  1.0         703            0.491
red_zone          0.0        5560            0.527
                  1.0        1873            0.532
short_yardage     0.0        3500            0.396
                  1.0        3933            0.646
end_of_half       0.0        5782            0.560
                  1.0        1651            0.417
late_game_deficit 0.0        5547            0.567
                  1.0        1886            0.416
overtime          0.0        7380            0.529
                  1.0          53            0.491

These rates are unadjusted: situation groups can differ in yards to go. Compare within distance groups before attributing a difference to the situation itself. Small groups have less reliable rates. Engineered features summarize existing information rather than introduce new observations; their usefulness depends on the model.

In [18]:
# Check goal-to-go and red-zone differences within distance groups
comparison = conversion_data.assign(distance_group=pd.cut(
    conversion_data["ydstogo"], bins=[0, 1, 2, 5, float("inf")],
    labels=["1 yard", "2 yards", "3–5 yards", "6+ yards"],
))
for column in ["goal_to_go", "red_zone"]:
    print(column)
    display(comparison.groupby(["distance_group", column], observed=True)["fourth_down_converted"].agg(
        attempts="size", conversion_rate="mean",
    ).round(3))

goal_to_go


attempts  conversion_rate
distance_group goal_to_go                           
1 yard         0.0             2562            0.684
               1.0              341            0.560
2 yards        0.0              907            0.582
               1.0              123            0.553
3–5 yards      0.0             1537            0.499
               1.0              140            0.421
6+ yards       0.0             1724            0.310
               1.0               99            0.273

red_zone


attempts  conversion_rate
distance_group red_zone                           
1 yard         0             1977            0.695
               1              926            0.616
2 yards        0              719            0.574
               1              311            0.588
3–5 yards      0             1293            0.502
               1              384            0.461
6+ yards       0             1571            0.314
               1              252            0.266

## 3. Dataset and modeling preparation

**Source:** [nflverse play-by-play](https://github.com/nflverse/nflverse-data/releases/tag/pbp), loaded through `nflreadpy` for the selected 2015–2025 NFL seasons, including regular-season and playoff games. Each raw row is a play-by-play record; each retained example is a fourth-down run/pass attempt with no flagged penalty and exactly one conversion outcome.

**Input:** the selected pre-snap situation, team, stadium/weather, season, and engineered features in `FEATURE_COLUMNS`. The exact list is displayed below. **Target:** `converted`, where 1 means successful conversion and 0 means failure. Original outcome fields remain in the processed table but are excluded from model inputs.

The shared functions now filter and engineer features, split examples 60/20/20 (train/test/validation) stratified by season with seed 42, and preprocess predictors. Numeric features are median-imputed and standardized; binary indicators remain 0/1 with mode imputation; categories receive a missing level and one-hot encoding. Preprocessing is fitted only on training data. The split is by play, so games may span partitions; earlier EDA used all selected seasons.

In [19]:
processed = process_4th_down_data(raw_data)
train, test, validation = split_data(processed, random_seed=42)

preprocessor = make_preprocessor()
X_train = preprocessor.fit_transform(train[FEATURE_COLUMNS])
X_valid = preprocessor.transform(validation[FEATURE_COLUMNS])
X_test = preprocessor.transform(test[FEATURE_COLUMNS])
y_train = train[TARGET_COLUMN]
y_valid = validation[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

print(f"Model inputs ({len(FEATURE_COLUMNS)} before encoding):")
print(", ".join(FEATURE_COLUMNS))

Model inputs (28 before encoding):
ydstogo, yardline_100, goal_to_go, qtr, game_seconds_remaining, half_seconds_remaining, posteam_timeouts_remaining, defteam_timeouts_remaining, score_differential, posteam, defteam, posteam_type, roof, surface, temp, wind, week, season_type, season, red_zone, short_yardage, end_of_half, late_game_deficit, overtime, log_ydstogo, distance_x_goal_to_go, distance_x_red_zone, timeout_advantage


In [20]:
print(f"Size of raw data: {raw_data.shape[0]:,} rows × {raw_data.shape[1]} columns")
print(f"Size of processed data: {processed.shape[0]:,} rows × {processed.shape[1]} columns")
print(f"Input features before encoding: {len(FEATURE_COLUMNS)}")
print(f"Input features after preprocessing: {X_train.shape[1]}")
print(f"Target: {TARGET_COLUMN} (1 = converted, 0 = failed)")
print(f"Training examples: {len(train):,}")
print(f"Validation examples: {len(validation):,}")
print(f"Test examples: {len(test):,}")

Size of raw data: 532,376 rows × 372 columns
Size of processed data: 7,433 rows × 382 columns
Input features before encoding: 28
Input features after preprocessing: 103
Target: converted (1 = converted, 0 = failed)
Training examples: 4,459
Validation examples: 1,487
Test examples: 1,487
